# 09 — Activity Continuity and Gaps

These reports describe recorded task starts, not health or productivity. Changes may reflect seasons, travel, or recording habits. Filters use exact names; `None` includes all values. Choose a task type to focus rolling and seasonal comparisons on one activity. Missing records are not proof of inactivity. Engagement-domain reports (including brief section 5) are deferred with the section 3 schema enhancement.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run reporting-utils.ipynb

import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))
from support.activity_analysis import (
    ReportPeriod,
    activity_continuity,
    filter_history,
)

In [ ]:
# Inclusive dates; END_DATE is also the as-of date. Use finite ISO dates.
START_DATE = "2024-01-01"
END_DATE = date.today().isoformat()
CATEGORY = None
TASK_TYPE = None
LOCATION = None
ROLLING_WINDOW = 90
DORMANCY_THRESHOLD = 90

In [ ]:
period = ReportPeriod(START_DATE, END_DATE, CATEGORY, TASK_TYPE, LOCATION)
# Load earlier observations for rolling windows, continuity and the full backlog.
query = construct_query("task-history.sql", {"START-DATE": "0001-01-01", "END-DATE": END_DATE})
history = filter_history(query_data(query), period)
filters = f"category={CATEGORY or 'All'}, type={TASK_TYPE or 'All'}, location={LOCATION or 'All'}"
period_label = f"{START_DATE} to {END_DATE}\n{filters}"
print(period_label)

## New, active, resumed and dormant task types

Status precedence: Dormant (at least two recorded active dates and absence ≥ threshold at period end), New (first recorded occurrence within period), Resumed (a gap ≥ threshold ends within period), Active (other starts in period), then Not active in period. “Active” describes occurrence, not a statistically inferred normal pattern. Previous Activity is the last date before the reporting period. First recorded does not necessarily mean first ever.


In [ ]:
continuity = activity_continuity(history, period, DORMANCY_THRESHOLD)
continuity

## Activity gaps

Days since last activity uses the end date. Longest and median gaps use consecutive active dates with both endpoints inside the selected period; fewer than two dates gives an undefined gap. Current gaps are reported separately, including types with no starts in the period.


In [ ]:
gaps = continuity[
    [
        "Category",
        "Task Type",
        "Latest Activity",
        "Days Since Last Activity",
        "Longest Gap (Days)",
        "Median Gap (Days)",
    ]
].copy()
gaps

In [ ]:
if not gaps.empty and len(gaps.columns) > 1:
    figure, axes = plt.subplots(
        figsize=(CHART_FIGURE_SIZE[0], max(CHART_FIGURE_SIZE[1], len(gaps) * 0.3))
    )
    gaps.assign(Activity=gaps["Category"] + " / " + gaps["Task Type"]).plot(
        x="Activity", y=["Days Since Last Activity"], kind="barh", stacked=False, ax=axes
    )
    format_chart(
        figure,
        axes,
        f"Time since last recorded activity — {period_label}",
        "Days",
        "Activity",
        grid_axis="x",
    )
    export_chart(get_export_folder_path(), "activity-gaps", "png", figure)
    plt.show()
else:
    print("No observations available for this chart.")

## Export

Tables are exported as XLSX and charts as PNG to `data/reports/` (or the runner’s configured destination).


In [ ]:
export_to_spreadsheet(
    get_export_folder_path(),
    "activity-continuity-and-gaps.xlsx",
    {"Activity Status": continuity, "Activity Gaps": gaps},
)